# Notebook 9 - DistilBERT Chatbot: Fine-tune a Transformer and Ship It

**Part C - Transfer Learning**
*Author: Axel Sirota, Data Trainers LLC*

## Story (STAR)

- **Situation.** In Notebook 8 you hand-built an MLP on averaged word2vec vectors. It classified
  review sentiment, but it plateaued. It reads "the update is not good" the same as "the update
  is good", because averaging word vectors throws away word order. The product team wants higher
  accuracy before this auto-tags incoming support messages.
- **Task.** Fine-tune a pretrained DistilBERT on the SST-2 sentiment benchmark, beat your MLP
  baseline on the same kind of binary task, then load the fine-tuned model into a simple Gradio
  chatbot a non-technical teammate can use.
- **Action.** Install HuggingFace `transformers`, `datasets`, `evaluate`, `accelerate`, and
  `gradio`. Tokenize with WordPiece via `AutoTokenizer`. Load `distilbert-base-uncased` with a
  2-class head. Fine-tune with the high-level `Trainer`. Evaluate on the held-out SST-2
  validation split. Save the model, reload it, and wrap it in a guarded Gradio interface.
- **Result.** Fine-tuned DistilBERT lands around 90 percent accuracy on SST-2 validation, above
  the averaged-embedding MLP, and it separates "not good" from "good". You ship a chatbot and a
  reusable recipe for fine-tuning any HuggingFace encoder.

## Learning objectives

By the end of this notebook you will be able to:

1. Explain in plain English what attention does and why it beats averaging word vectors.
2. Describe DistilBERT in one breath: encoder-only, 6 layers, 66M params, bidirectional context.
3. Tokenize text with WordPiece subwords and read `input_ids`, `attention_mask`, special tokens.
4. Fine-tune `distilbert-base-uncased` for sequence classification with the HuggingFace `Trainer`.
5. Evaluate against your MLP baseline and decide when a transformer is worth its cost.
6. Save, reload, and wrap the model in a Gradio chatbot.

## Prerequisites

Notebooks B6-B8 (PyTorch nn, MLP on word2vec, Capstone B). You have `device`, `SEED = 42`, and a
working mental model of "embeddings as features". GPU recommended (Runtime -> Change runtime type
-> T4 GPU on Colab); the notebook still runs on CPU, just slower.

## Section 0: Environment Setup

Install dependencies. This pulls `transformers`, `datasets`, `evaluate`, `accelerate`, `gradio`,
and pins `numpy<2`. On a fresh Colab session budget 2-4 minutes the first time.

> **Colab restart note.** Colab now ships NumPy 2.x preinstalled. Because parts of the stack are
> compiled against NumPy 1.x, we pin `numpy<2`. After the install cell finishes you may see
> "You must restart the runtime to use newly installed versions." If so, click **Runtime ->
> Restart runtime**, then run the cells again from the top (skip nothing).

> **Heads-up:** The first call to `from_pretrained('distilbert-base-uncased')` downloads about
> 270 MB of weights from the HuggingFace Hub and caches them under `~/.cache/huggingface/`.

In [ ]:
# Install the fine-tuning + chatbot stack, pinned to versions verified for this course.
# - transformers 4.57.1: AutoTokenizer, AutoModelForSequenceClassification, Trainer.
#   Pinned below 5.x on purpose; 5.x is a breaking release.
# - datasets <3: load_dataset('glue', 'sst2') and .map() tokenization.
# - evaluate: accuracy / F1 metrics.
# - accelerate: required by Trainer for device placement and fp16.
# - gradio: the chatbot UI at the end.
# - scikit-learn: confusion matrix and classification report.
# - numpy<2: avoid the "ndarray size changed" binary-incompatibility error.
!pip install -q "transformers==4.57.1" "datasets>=2.19,<3" "evaluate>=0.4" \
    "accelerate>=0.28" "gradio>=4.0" "scikit-learn>=1.3" "numpy<2"

# If Colab warns about restarting the runtime, do it now (Runtime -> Restart runtime),
# then re-run from the top.

In [ ]:
# Imports in the course order: data -> model -> training -> evaluation.
import os
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
    pipeline,
)
from datasets import load_dataset
import evaluate

from sklearn.metrics import confusion_matrix, classification_report

warnings.filterwarnings("ignore")

# Reproducibility: same SEED convention as every notebook in this course.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Device auto-select, same object you have used since B4.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"transformers ready | torch {torch.__version__} | device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# All hyperparameters up front (course convention), so you can tune in one place.
MODEL_NAME   = "distilbert-base-uncased"  # encoder-only, 6 layers, ~66M params, Colab-friendly
NUM_LABELS   = 2        # SST-2 is binary: 0 = negative, 1 = positive
MAX_LENGTH   = 64       # SST-2 sentences are short; 64 subword tokens is plenty
LR           = 2e-5     # canonical transformer fine-tune LR (about 100x smaller than an MLP LR)
NUM_EPOCHS   = 2        # 2-3 epochs is enough for GLUE; more overfits
BATCH_SIZE   = 32       # fits a T4 with fp16; halve if you hit out-of-memory
WEIGHT_DECAY = 0.01     # AdamW decoupled weight decay
TRAIN_SUBSET = 6000     # subsample train for class-time speed (full SST-2 is 67k rows)
FP16         = torch.cuda.is_available()  # half precision only makes sense on GPU

# Label names so predictions read as words, not integers.
id2label = {0: "NEGATIVE", 1: "POSITIVE"}
label2id = {"NEGATIVE": 0, "POSITIVE": 1}

print(f"Will fine-tune '{MODEL_NAME}' for {NUM_EPOCHS} epochs at LR={LR} on {NUM_LABELS} classes.")

## Section 1: Why Transformers? The MLP Ceiling

In Notebook 8 you averaged the word2vec vectors of a review into one document vector, then fed
that to an MLP. It works, but it has a hard ceiling. Consider two reviews:

1. *"the update is good"*
2. *"the update is not good"*

To a human these are opposites. To your averaged-embedding MLP they are almost the same vector:
mean pooling adds up the same word vectors, and the tiny vector for "not" barely moves the
average. The MLP literally cannot see that "not" flips the meaning, because **averaging destroys
word order**.

A transformer fixes this two ways:

1. **Subword tokenization.** "disappointing" becomes `['disappoint', '##ing']`, so the model
   shares the root "disappoint" across "disappointed", "disappoints", "disappointing".
2. **Contextual embeddings via attention.** The representation of "good" in sentence 2 is
   different from "good" in sentence 1, because the model lets every token look at every other
   token and re-weight them. "good" attends to "not" and learns the phrase is negative.

That is why a fine-tuned BERT-family model beats a static-embedding MLP on sentiment, where
word order and negation decide the label.

**Diagram: averaging destroys word order, attention reads context**

```mermaid
graph TD
    A["the update is not good"] --> B["MLP path: average word vectors"]
    A --> C["Transformer path: attention"]
    B --> D["mean pooling drops word order"]
    D --> E["not barely moves the average"]
    E --> F["predicts POSITIVE (wrong)"]
    C --> G["good attends to not"]
    G --> H["context re-weights each token"]
    H --> I["predicts NEGATIVE (correct)"]
```


### Attention in plain English (no math)

Attention lets every token in a sentence look at every other token and decide how much each one
should contribute to its own updated meaning. In a plain feed-forward net each token's
representation is fixed; in a transformer it is recomputed, layer by layer, as a learned weighted
mix of all positions.

Example: in *"The update shipped late. **It** broke login."*, the token "it" should attend
strongly to "update" to know what "it" refers to. The model learns those weights from data.

The takeaway: every token's hidden state is a learned weighted blend of all the other tokens.
That is how the model "reads context". For classification we add a special `[CLS]` token at
position 0; after the forward pass its hidden state has attended to the whole sentence, and we
feed that one vector into a linear classifier head.

### DistilBERT at a glance

**DistilBERT** is a distilled (compressed) version of BERT:

- **Encoder-only.** It sees the whole sentence at once (bidirectional), unlike a decoder-only
  GPT model that reads strictly left to right.
- **Pretrained** by self-supervision (masked-language modeling: hide 15 percent of tokens and
  predict them). That is where its language knowledge comes from, for free, before you ever
  fine-tune.
- **6 transformer layers** (BERT-base has 12), about **66M parameters**, 40 percent smaller and
  60 percent faster than BERT-base while keeping roughly 97 percent of its quality. That is why
  it fits class time.

Roadmap for this notebook: tokenize -> load the pretrained model with a 2-class head ->
fine-tune with `Trainer` -> evaluate vs the MLP -> save, reload, and ship a Gradio chatbot.

In [ ]:
# Load the tokenizer that matches our model. AutoTokenizer picks WordPiece for DistilBERT.
# First call downloads ~470 KB of vocab files.
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"Tokenizer class: {tokenizer.__class__.__name__} | vocab size: {len(tokenizer)}")

# Tokenize one sentiment sentence and look at what BERT actually sees.
demo_text = "this update is unbelievably disappointing"
demo = tokenizer(demo_text)
print(f"\nText  : {demo_text}")
print(f"IDs   : {demo['input_ids']}")
print(f"Tokens: {tokenizer.convert_ids_to_tokens(demo['input_ids'])}")

# Subword splitting in action: rare words break into pieces with a '##' continuation prefix.
for word in ["unbelievably", "disappointing", "soaring"]:
    print(f"  '{word}' -> {tokenizer.tokenize(word)}")
# '##' means "glue this onto the previous piece". This is how WordPiece shares roots across
# inflections and still represents words it never saw whole.

**Diagram: encoder-only vs decoder-only vs encoder-decoder**

```mermaid
graph TD
    subgraph EncoderOnly["Encoder-only (DistilBERT)"]
        E1["whole sentence at once"] --> E2["bidirectional context"]
        E2 --> E3["best for classification"]
    end
    subgraph DecoderOnly["Decoder-only (GPT)"]
        D1["reads left to right"] --> D2["predicts next token"]
        D2 --> D3["best for text generation"]
    end
    subgraph EncDec["Encoder-decoder (T5 / BART)"]
        ED1["encode input"] --> ED2["decode output"]
        ED2 --> ED3["best for seq-to-seq"]
    end
```


### What the tokenizer returns

The tokenizer hands back a dictionary. The two fields that matter for us:

- **`input_ids`**: the subword IDs. They always start with `[CLS]` (ID 101 for DistilBERT) and
  end with `[SEP]` (ID 102). You saw both in the cell above.
- **`attention_mask`**: a 1/0 mask, 1 for real tokens and 0 for padding. The model ignores every
  position where the mask is 0. This is how a batch of different-length sentences becomes one
  rectangular tensor: pad the short ones, then mask the padding out.

Why subwords are worth it:

1. They handle words the model never saw whole (split into known pieces).
2. They share roots across inflections ("disappoint" inside "disappointing").
3. They keep the vocabulary small (about 30K subwords instead of 100K+ full words).

About `max_length`: longer sequences cost more memory and time. SST-2 sentences are short, so 64
tokens is plenty. Rule of thumb: tweets/headlines 32-64, paragraphs 128-256, long documents 512
(DistilBERT's hard limit).

**Diagram: WordPiece tokenization into input_ids and attention_mask**

```mermaid
graph TD
    A["raw text: this is disappointing"] --> B["WordPiece split"]
    B --> C["disappoint + ##ing"]
    C --> D["add CLS at front, SEP at end"]
    D --> E["input_ids (subword IDs)"]
    D --> F["attention_mask (1 real, 0 pad)"]
    E --> G["padded rectangular tensor"]
    F --> G
    G --> H["fed to DistilBERT"]
```


In [ ]:
# Special token IDs, confirmed: [CLS]=101, [SEP]=102, [PAD]=0.
print("Special tokens:")
for name, tok, tid in [
    ("CLS", tokenizer.cls_token, tokenizer.cls_token_id),
    ("SEP", tokenizer.sep_token, tokenizer.sep_token_id),
    ("PAD", tokenizer.pad_token, tokenizer.pad_token_id),
]:
    print(f"  [{name}] = {tok} -> id {tid}")

# Tokenize a small batch with padding + truncation, returning PyTorch tensors.
# padding=True   -> pad every sequence to the longest one in this batch.
# truncation=True-> cut anything longer than max_length.
batch_texts = [
    "great product",
    "the support team was helpful and quick",
    "honestly the worst experience i have had with any service this year",
]
enc = tokenizer(batch_texts, padding=True, truncation=True, max_length=32, return_tensors="pt")
print(f"\ninput_ids shape: {tuple(enc['input_ids'].shape)}  (rows=sentences, cols=padded length)")
print(f"first row ids : {enc['input_ids'][0].tolist()}")
print(f"first row mask: {enc['attention_mask'][0].tolist()}  (1=real token, 0=padding)")

### Lab 1: Read what the tokenizer does (15 min)

Tokenize five sentences of varying length and report two numbers, so you build intuition for
padding and truncation before you tokenize a whole dataset.

**Your task:**

1. Write a list of 5 sentences of clearly different lengths (a 2-word one, a medium one, a long
   one, and so on). Sentiment-flavoured is fine.
2. Tokenize them together with truncation, `max_length=32`, and PyTorch tensors. Pad every row
   to the full `max_length` (a fixed width of 32 columns, not just to the batch's longest row),
   so every row is exactly 32 wide and the padding ratio below is well defined.
3. From the result, find which sentence has the **most real (non-padding) tokens**, and compute
   the **padding ratio** of the shortest sentence: number of padding positions divided by 32.

**Verification** (provided) prints the shapes and your two numbers so you can sanity-check.

In [ ]:
# LAB 1: Tokenization analysis.

# 1. Five sentences of clearly different lengths.
lab_sentences = None  # YOUR CODE (a list of five strings, short to long)

# 2. Tokenize them together so they come back as one fixed-width PyTorch tensor batch.
#    Hint: pass the list to the tokenizer and ask it to truncate, return tensors, cap length at
#    32, and pad every row to that full fixed width (not just to the batch's longest row) so the
#    pad-ratio denominator below is exactly 32. The call returns a dict with the ids and the mask.
lab_enc = None  # YOUR CODE

# 3. Using lab_enc, compute:
#    real_counts      -> for each sentence, how many positions are real (not padding).
#                        Hint: the mask field is 1 on real tokens; a row-wise total gives the count.
#    longest_idx      -> index of the sentence with the most real tokens.
#    shortest_pad_ratio -> for the sentence with the fewest real tokens, the fraction of its 32
#                        positions that are padding.
real_counts = None         # YOUR CODE
longest_idx = None         # YOUR CODE
shortest_pad_ratio = None  # YOUR CODE

# ---- Verification (provided) ----
if lab_enc is not None and real_counts is not None:
    print(f"input_ids shape    : {tuple(lab_enc['input_ids'].shape)}")
    print(f"real-token counts  : {list(map(int, real_counts))}")
    print(f"longest sentence   : index {int(longest_idx)} -> '{lab_sentences[int(longest_idx)]}'")
    print(f"shortest pad ratio : {float(shortest_pad_ratio):.2f}")

## Section 2: Load the SST-2 Sentiment Dataset

We use **SST-2** (Stanford Sentiment Treebank, the GLUE binary version): single sentences
labelled 0 = negative, 1 = positive. It is the standard sentiment benchmark, and it is the same
kind of binary sentiment task your MLP did in Notebook 8, so the comparison is fair.

One gotcha to know up front: GLUE's official **test split has hidden labels** (every label is
`-1`) because the real answers live on a private leaderboard. So we do the standard thing:
**train on the train split and evaluate on the validation split** (872 labelled sentences). We
also subsample the training set to a few thousand rows so fine-tuning finishes inside class time;
the full set is about 67K rows.

In [ ]:
# Load SST-2 from the HuggingFace Hub. Fields: 'sentence', 'label', 'idx'.
raw = load_dataset("glue", "sst2")
print(raw)  # DatasetDict with train (67349), validation (872), test (1821, labels = -1)

# Subsample the train split for class-time speed; shuffle with our SEED for reproducibility.
train_ds = raw["train"].shuffle(seed=SEED).select(range(TRAIN_SUBSET))
eval_ds  = raw["validation"]   # 872 labelled rows -> this is our held-out test set

# IMPORTANT gotcha: the model's forward() looks for a column literally named 'labels' to compute
# the loss. SST-2 calls it 'label'. If you skip this rename the Trainer trains on no labels and
# silently learns nothing. Rename it now.
train_ds = train_ds.rename_column("label", "labels")
eval_ds  = eval_ds.rename_column("label", "labels")

print(f"\nTrain rows: {len(train_ds)} | Eval (validation) rows: {len(eval_ds)}")
print(f"Example: {train_ds[0]}")

### Lab 2: Wire the tokenization step (Tier 2, 12 min)

The fine-tuning pipeline has one piece you have not written yet: turning raw `sentence` strings
into `input_ids` and `attention_mask` for every row, in one pass. The `Trainer` in the next
section reads those columns straight off `train_ds` and `eval_ds`, so this is the bridge between
the dataset and the model.

**Your task:**

1. Write `tokenize_fn(examples)` so it tokenizes the batch's `sentence` field. Truncate to
   `MAX_LENGTH`. Do NOT pad here: padding is done dynamically per batch by the collator below, so
   passing `padding=...` now would waste memory.
2. Apply it to both `train_ds` and `eval_ds` in one pass each (batched), reassigning each.
3. The dynamic-padding collator is provided.

You are combining two ideas you have already seen: the tokenizer call from the demo cells, and
the `.map(batched=True)` pattern. There is no `None # YOUR CODE` scaffold inside the function body
this time; recall the demo and write it.

**Verification** (provided, after the safety-net cell) confirms `input_ids` and `attention_mask`
landed on the dataset and that `labels` survived.

In [ ]:
# LAB 2 (Tier 2): wire the tokenization step of the fine-tuning pipeline.

# 1. Tokenize the batch's sentences. Truncate to MAX_LENGTH; do NOT pad here (the collator pads
#    dynamically per batch). Return the tokenizer's output directly.
def tokenize_fn(examples):
    """Tokenize one batch of SST-2 rows: read the 'sentence' field, truncate to MAX_LENGTH,
    no padding. Return the tokenizer output (a dict with input_ids and attention_mask)."""
    pass  # YOUR CODE

# 2. Apply tokenize_fn to both splits in one pass each (batched), reassigning each dataset.
train_ds = train_ds  # YOUR CODE: map tokenize_fn over train_ds, batched
eval_ds  = eval_ds   # YOUR CODE: map tokenize_fn over eval_ds, batched

# 3. The collator pads each batch to that batch's longest sequence, not the global max. (Provided.)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

<details>
<summary>Stuck? Reveal the safety-net</summary>

```python
def tokenize_fn(examples):
    return tokenizer(examples["sentence"], truncation=True, max_length=MAX_LENGTH)

train_ds = train_ds.map(tokenize_fn, batched=True)
eval_ds  = eval_ds.map(tokenize_fn, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
```
</details>

In [ ]:
# ---- Silent rescue (provided): keep the pipeline runnable even if Lab 2 above was skipped ----
# The Trainer in Section 4 reads input_ids / attention_mask off train_ds and eval_ds. If Lab 2 was
# left unfinished those columns are missing, so this guard tokenizes for you with a known-good
# fallback (revealed in the collapsed block above) so the rest of the notebook still runs. It does
# nothing if Lab 2 already produced the columns.
if "input_ids" not in train_ds.column_names:
    train_ds = train_ds.map(lambda ex: tokenizer(ex["sentence"], truncation=True, max_length=MAX_LENGTH), batched=True)
    eval_ds  = eval_ds.map(lambda ex: tokenizer(ex["sentence"], truncation=True, max_length=MAX_LENGTH), batched=True)

# Make sure the collator exists too (Lab 2 step 3 defines it; provide it if missing).
try:
    data_collator
except NameError:
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# ---- Verification (provided, runs AFTER the rescue so an empty Lab 2 does not crash) ----
print(f"Tokenized columns: {train_ds.column_names}")
print("Expected to include: input_ids, attention_mask, and labels.")
assert "input_ids" in train_ds.column_names and "attention_mask" in train_ds.column_names
assert "labels" in train_ds.column_names

## Section 3: Load the Pretrained Model

`AutoModelForSequenceClassification` loads the pretrained DistilBERT backbone and bolts a fresh
classification head on top:

```
input -> DistilBERT (6 layers) -> [CLS] hidden state -> Dropout -> Linear(768 -> 2)
```

The **backbone** already knows English from pretraining. The **head** (the final Linear layer)
is **randomly initialized** and gets trained from scratch on sentiment.

Fine-tuning vs training from scratch:

- **From scratch**: every weight random, needs millions of labelled examples. Only big labs do
  this.
- **Fine-tuning**: start from pretrained weights, nudge them on your task. Works with thousands
  of examples. The head trains fully; the backbone updates gently at LR 2e-5 so we do not wipe
  out the pretrained knowledge (catastrophic forgetting).

You will see a warning that some weights are "newly initialized". That is expected: it is the new
classification head.

In [ ]:
# Load DistilBERT with a 2-class head. First call downloads ~270 MB of weights.
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=id2label,   # so predictions come back as POSITIVE / NEGATIVE
    label2id=label2id,
).to(device)

print(f"Loaded {model.__class__.__name__} | params: {sum(p.numel() for p in model.parameters()):,}")

# Sanity check: BEFORE training, the head is random, so it should guess near 50/50.
model.eval()
probe = tokenizer(["this is wonderful", "this is terrible"],
                  padding=True, truncation=True, max_length=MAX_LENGTH, return_tensors="pt").to(device)
with torch.no_grad():
    probs = torch.softmax(model(**probe).logits, dim=-1)
for text, p in zip(["this is wonderful", "this is terrible"], probs):
    print(f"  '{text}' -> {id2label[int(p.argmax())]} (probs {p.cpu().numpy().round(3)})  # untrained = near random")

## Section 4: Fine-tune with the Trainer

HuggingFace `Trainer` is the high-level wrapper that runs the loop you built by hand in B6 and
B7: forward, loss, `backward()`, optimizer step, evaluation, checkpointing, fp16. You already
know what it does under the hood, so here we just configure it and call `train()`.

You configure three things: a `compute_metrics` function (so it reports accuracy and F1), a
`TrainingArguments` object (epochs, batch size, learning rate, when to evaluate), and the
`Trainer` itself (model + data + tokenizer + collator + metrics). Then `trainer.train()`.

**Diagram: the fine-tuning loop the Trainer runs for you**

```mermaid
graph TD
    A["pretrained DistilBERT + fresh 2-class head"] --> B["forward pass on a batch"]
    B --> C["compute loss vs labels"]
    C --> D["backward pass"]
    D --> E["AdamW step at LR 2e-5"]
    E --> F{"epoch done?"}
    F -->|no| B
    F -->|yes| G["evaluate accuracy and F1"]
    G --> H{"all epochs done?"}
    H -->|no| B
    H -->|yes| I["reload best checkpoint"]
```


In [ ]:
# 1. Metrics the Trainer will compute on the eval split after each epoch.
accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "f1": f1.compute(predictions=preds, references=labels, average="macro")["f1"],
    }

# 2. Training configuration.
#    eval_strategy='epoch'   -> evaluate after every epoch (4.46+ name; 'evaluation_strategy' is gone).
#    load_best_model_at_end  -> after training, reload the checkpoint with the best F1.
#    fp16=FP16               -> half precision, GPU only (it errors on CPU, hence the guard).
#    use_cpu=True off-GPU    -> on a non-CUDA box (e.g. Apple Silicon) pin the Trainer to CPU so it
#                               does not pick MPS, where some fine-tuning ops still crash. The CUDA
#                               fast path on Colab is untouched.
USE_CPU = not torch.cuda.is_available()
training_args = TrainingArguments(
    output_dir="./distilbert_sst2",
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LR,
    weight_decay=WEIGHT_DECAY,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=50,
    fp16=FP16,
    use_cpu=USE_CPU,
    seed=SEED,
    report_to="none",
)

# 3. The Trainer. processing_class=tokenizer is the current name (the old 'tokenizer=' is deprecated).
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# 4. Fine-tune. ~3-6 min on a Colab T4; much slower on CPU. Watch loss fall and F1 rise.
trainer.train()
print("\nFine-tuning done. Best checkpoint reloaded automatically.")

In [ ]:
# Evaluate the fine-tuned model on the held-out validation split (our test set).
results = trainer.evaluate(eval_ds)
print("=== DistilBERT on SST-2 validation ===")
print(f"Accuracy: {results['eval_accuracy']:.4f}")
print(f"Macro-F1: {results['eval_f1']:.4f}")

# Compare to your MLP from Notebook 8. The averaged-embedding MLP sat in the low-to-mid 80s on
# binary sentiment; fine-tuned DistilBERT should land around 0.90. The gain comes from attention
# (it reads "not good" as negative) plus pretraining. That is the MLP ceiling, broken.
#
# Cost side of the trade: DistilBERT is ~270 MB and ~10-20 ms per sentence on GPU, versus a
# ~2 MB MLP at well under 1 ms. You buy accuracy with size and latency.

# ---------------------------------------------------------------------------------------------
# STRETCH (fast finishers, ~10 min): freeze the backbone, train only the head.
# Production teams sometimes freeze the pretrained layers and train just the classifier: far
# faster and lighter, usually a few points less accurate. Try it and measure the gap.
#
#   frozen = AutoModelForSequenceClassification.from_pretrained(
#       MODEL_NAME, num_labels=NUM_LABELS, id2label=id2label, label2id=label2id).to(device)
#   for p in frozen.distilbert.parameters():   # freeze the 6-layer backbone
#       p.requires_grad = False                # keep frozen.classifier trainable
#   # Build a second Trainer with `model=frozen` and the SAME args/data, then train + evaluate.
#   # Question to answer: how many accuracy points did you give up to gain the speed?
# ---------------------------------------------------------------------------------------------

In [ ]:
# Quick human-readable inference with a pipeline. Note: pipeline's device wants an INT
# (0 = first GPU, -1 = CPU), not a torch.device object, so convert.
clf = pipeline("text-classification", model=model, tokenizer=tokenizer,
               device=0 if torch.cuda.is_available() else -1)

probe_texts = [
    "this is the best support i have ever received",
    "the app keeps crashing and nobody replies",
    "the update is good",
    "the update is not good",   # negation: the MLP missed this; the transformer should not
]
for t in probe_texts:
    out = clf(t, top_k=1)[0]
    print(f"{t:<52} -> {out['label']} ({out['score']:.3f})")

# Confusion matrix on the validation set, straight from trainer.predict (no manual loop needed).
pred = trainer.predict(eval_ds)
y_pred = np.argmax(pred.predictions, axis=-1)
y_true = pred.label_ids

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["NEGATIVE", "POSITIVE"], yticklabels=["NEGATIVE", "POSITIVE"], square=True)
plt.xlabel("Predicted"); plt.ylabel("True"); plt.title("DistilBERT SST-2 Confusion Matrix")
plt.tight_layout(); plt.show()

print(classification_report(y_true, y_pred, target_names=["NEGATIVE", "POSITIVE"]))
# Off-diagonal cells are the mistakes. Many will be genuinely ambiguous one-liners; a systematic
# block (e.g. all positives called negative) would mean a bug in labels or the rename step.

In [ ]:
# Save the fine-tuned model and tokenizer. save_pretrained writes config.json (which carries our
# id2label map) plus model.safetensors (the weights). The chatbot will reload from this folder.
SAVE_DIR = "./distilbert_sst2_final"
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f"Saved to {SAVE_DIR}/  ->  {sorted(os.listdir(SAVE_DIR))}")

# Reload to prove the round trip works and that labels come back as words (from config.json).
reloaded = AutoModelForSequenceClassification.from_pretrained(SAVE_DIR).to(device)
reloaded_tok = AutoTokenizer.from_pretrained(SAVE_DIR)
check = tokenizer("support was fantastic today", return_tensors="pt").to(device)
reloaded.eval()
with torch.no_grad():
    pid = int(reloaded(**check).logits.argmax(-1))
print(f"Reloaded model predicts: {reloaded.config.id2label[pid]}")

## Section 5: Ship It - A Gradio Chatbot

Time to deliver the artifact this whole course builds toward. We wrap the reloaded model in a
`pipeline`, then hand that pipeline to Gradio, which auto-builds a textbox-in / label-out web UI.
A teammate types a support message; the model returns POSITIVE or NEGATIVE with a confidence.

We guard the UI in `try/except ImportError` so the notebook still runs end to end in an
environment without Gradio. On Colab, `.launch()` prints a public share link automatically.

**Diagram: save, reload, pipeline, and the Gradio chatbot path**

```mermaid
graph TD
    A["fine-tuned model"] --> B["save_pretrained: config.json + safetensors"]
    B --> C["from_pretrained reloads weights + id2label"]
    C --> D["build text-classification pipeline"]
    D --> E["wrap in gradio Interface"]
    E --> F["teammate types a message"]
    F --> G["pipeline classifies"]
    G --> H["POSITIVE or NEGATIVE + confidence"]
```


In [ ]:
# Build the inference pipeline from the reloaded, saved model (same recipe a real service uses).
chat_pipe = pipeline("text-classification", model=SAVE_DIR,
                     device=0 if torch.cuda.is_available() else -1)

def classify_message(text):
    """Return a human-readable verdict for one message."""
    if not text or not text.strip():
        return "Type a message to classify."
    out = chat_pipe(text, top_k=1)[0]
    return f"{out['label']}  (confidence {out['score']:.2f})"

# Quick sanity check that works with or without Gradio installed.
print(classify_message("the new release fixed my problem instantly"))
print(classify_message("i have been waiting three days for a reply"))

# Launch the chatbot UI, guarded so a no-Gradio environment still runs everything above.
try:
    import gradio as gr
    demo = gr.Interface(
        fn=classify_message,
        inputs=gr.Textbox(lines=2, placeholder="Type a support message..."),
        outputs=gr.Textbox(label="Sentiment"),
        title="DistilBERT Sentiment Chatbot",
        description="Fine-tuned distilbert-base-uncased on SST-2. POSITIVE or NEGATIVE with confidence.",
    )
    # On Colab a public share link is created automatically. share=True forces it elsewhere.
    demo.launch(share=True)
except ImportError:
    print("\n[Gradio not installed] Skipping the UI. classify_message() still works above.")

## Wrap-up

### What you can now ship

- A fine-tuned transformer sentiment classifier that beats your averaged-embedding MLP.
- A reusable recipe: swap `MODEL_NAME` and `num_labels` to fine-tune any HuggingFace encoder
  (RoBERTa, DeBERTa, ELECTRA) on any classification task.
- A working Gradio chatbot wrapping your model, the course's end-deliverable shape.

### Key takeaways

| Concept | Remember |
|---------|----------|
| Attention | Tokens re-weight each other's features, so "good" and "not good" differ. |
| WordPiece | Splits rare words into `##` subwords and shares roots across inflections. |
| Fine-tune LR | 2e-5 is standard, about 100x smaller than an MLP LR. Too high -> NaN / no convergence. |
| Epochs | 2-3 for GLUE; 6+ overfits. |
| labels column | The model wants a column named `labels`; rename SST-2's `label` or it learns nothing. |
| Trainer | High-level loop; `eval_strategy`, `processing_class` are the current arg names. |
| When BERT | >1K labels, context/negation matters, GPU budget. Else keep the MLP (sub-ms, 2 MB). |

### Homework (async, deeper)

Add a **confidence gate** to the chatbot for the support-platform. Compute the softmax
confidence of each prediction; if it is below a threshold (say 0.65), do not auto-tag. Return
"uncertain - route to a human agent" instead. This is the standard selective-prediction / reject
-option pattern: the model answers only when confident and defers the rest to a person. Measure
how coverage (fraction auto-handled) trades off against accuracy as you move the threshold.

Production take-homes worth reading: quantize the model to int8 (about 4x smaller, 2-3x faster,
negligible accuracy loss on classification); batch your inputs instead of one at a time;
host the Gradio app on a HuggingFace Space; monitor average confidence over time to catch drift.

### Next: C10

Classification gives a label. A real assistant *writes* an answer. In C10 you fine-tune an
encoder-decoder model (T5 / BART) on a Q&A dataset and drop it into this same Gradio shell, so
the chatbot generates answers instead of picking a class.